<a href="https://colab.research.google.com/github/saurav2006-cyber/001/blob/main/Sudoku_Solver_and_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random
import copy
import time
from typing import List, Tuple, Optional

class Sudoku:
    def __init__(self):
        self.size = 9
        self.box_size = 3

    def print_board(self, board: List[List[int]]) -> None:
        """Print the Sudoku board in a readable format"""
        for i in range(self.size):
            if i % 3 == 0 and i != 0:
                print("-" * 21)
            for j in range(self.size):
                if j % 3 == 0 and j != 0:
                    print("|", end=" ")
                print(board[i][j] if board[i][j] != 0 else ".", end=" ")
            print()

    def is_valid(self, board: List[List[int]], num: int, pos: Tuple[int, int]) -> bool:
        """Check if placing a number at given position is valid"""
        row, col = pos

        # Check row
        for j in range(self.size):
            if board[row][j] == num and j != col:
                return False

        # Check column
        for i in range(self.size):
            if board[i][col] == num and i != row:
                return False

        # Check 3x3 box
        box_x = col // 3
        box_y = row // 3

        for i in range(box_y * 3, box_y * 3 + 3):
            for j in range(box_x * 3, box_x * 3 + 3):
                if board[i][j] == num and (i, j) != pos:
                    return False

        return True

    def find_empty(self, board: List[List[int]]) -> Optional[Tuple[int, int]]:
        """Find an empty cell (0) in the board"""
        for i in range(self.size):
            for j in range(self.size):
                if board[i][j] == 0:
                    return (i, j)
        return None

    def solve(self, board: List[List[int]]) -> bool:
        """Solve Sudoku using backtracking algorithm"""
        empty = self.find_empty(board)

        if not empty:
            return True

        row, col = empty

        for num in range(1, 10):
            if self.is_valid(board, num, (row, col)):
                board[row][col] = num

                if self.solve(board):
                    return True

                board[row][col] = 0

        return False

    def generate_complete_board(self) -> List[List[int]]:
        """Generate a complete, valid Sudoku board"""
        board = [[0 for _ in range(self.size)] for _ in range(self.size)]

        # Fill diagonal 3x3 boxes (they are independent)
        for i in range(0, self.size, 3):
            self.fill_box(board, i, i)

        # Solve the rest of the board
        self.solve(board)
        return board

    def fill_box(self, board: List[List[int]], row: int, col: int) -> None:
        """Fill a 3x3 box with random numbers"""
        numbers = list(range(1, 10))
        random.shuffle(numbers)

        for i in range(3):
            for j in range(3):
                board[row + i][col + j] = numbers.pop()

    def remove_numbers(self, board: List[List[int]], difficulty: str = "medium") -> List[List[int]]:
        """Remove numbers from complete board to create a puzzle"""
        difficulty_levels = {
            "easy": 35,      # Remove 35-40 numbers
            "medium": 45,    # Remove 45-50 numbers
            "hard": 55,      # Remove 55-60 numbers
            "expert": 60     # Remove 60+ numbers
        }

        cells_to_remove = difficulty_levels.get(difficulty, 45)
        puzzle = copy.deepcopy(board)
        cells = [(i, j) for i in range(self.size) for j in range(self.size)]
        random.shuffle(cells)

        removed = 0
        for row, col in cells:
            if removed >= cells_to_remove:
                break

            # Store the original value
            original = puzzle[row][col]
            puzzle[row][col] = 0

            # Check if puzzle still has unique solution
            temp_board = copy.deepcopy(puzzle)
            if not self.has_unique_solution(temp_board):
                puzzle[row][col] = original
            else:
                removed += 1

        return puzzle

    def has_unique_solution(self, board: List[List[int]]) -> bool:
        """Check if the puzzle has exactly one solution"""
        board_copy = copy.deepcopy(board)
        solutions = []
        self.count_solutions(board_copy, solutions)
        return len(solutions) == 1

    def count_solutions(self, board: List[List[int]], solutions: list, max_solutions: int = 2) -> None:
        """Count number of solutions (stop after max_solutions)"""
        if len(solutions) >= max_solutions:
            return

        empty = self.find_empty(board)
        if not empty:
            solutions.append(copy.deepcopy(board))
            return

        row, col = empty
        for num in range(1, 10):
            if self.is_valid(board, num, (row, col)):
                board[row][col] = num
                self.count_solutions(board, solutions, max_solutions)
                board[row][col] = 0

                if len(solutions) >= max_solutions:
                    return

    def generate_puzzle(self, difficulty: str = "medium") -> Tuple[List[List[int]], List[List[int]]]:
        """Generate a Sudoku puzzle and its solution"""
        solution = self.generate_complete_board()
        puzzle = self.remove_numbers(solution, difficulty)
        return puzzle, solution

    def validate_puzzle(self, puzzle: List[List[int]]) -> bool:
        """Validate if a puzzle is solvable and has unique solution"""
        # Check initial validity
        for i in range(self.size):
            for j in range(self.size):
                if puzzle[i][j] != 0:
                    if not self.is_valid(puzzle, puzzle[i][j], (i, j)):
                        return False

        # Check if solvable and has unique solution
        return self.has_unique_solution(copy.deepcopy(puzzle))

# Example usage and testing
def main():
    sudoku = Sudoku()

    print("=== SUDOKU SOLVER AND GENERATOR ===\n")

    # Example: Solve a puzzle
    print("1. SOLVING A PUZZLE:")
    puzzle = [
        [5, 3, 0, 0, 7, 0, 0, 0, 0],
        [6, 0, 0, 1, 9, 5, 0, 0, 0],
        [0, 9, 8, 0, 0, 0, 0, 6, 0],
        [8, 0, 0, 0, 6, 0, 0, 0, 3],
        [4, 0, 0, 8, 0, 3, 0, 0, 1],
        [7, 0, 0, 0, 2, 0, 0, 0, 6],
        [0, 6, 0, 0, 0, 0, 2, 8, 0],
        [0, 0, 0, 4, 1, 9, 0, 0, 5],
        [0, 0, 0, 0, 8, 0, 0, 7, 9]
    ]

    print("Original Puzzle:")
    sudoku.print_board(puzzle)

    if sudoku.solve(puzzle):
        print("\nSolved Puzzle:")
        sudoku.print_board(puzzle)
    else:
        print("\nNo solution exists!")

    # Example: Generate puzzles
    print("\n" + "="*50)
    print("2. GENERATING PUZZLES:")

    difficulties = ["easy", "medium", "hard"]

    for difficulty in difficulties:
        print(f"\n{difficulty.upper()} puzzle:")
        puzzle, solution = sudoku.generate_puzzle(difficulty)
        sudoku.print_board(puzzle)

        # Verify the generated puzzle
        is_valid = sudoku.validate_puzzle(puzzle)
        print(f"Valid puzzle: {is_valid}")

# Performance testing
def benchmark():
    """Benchmark the solver performance"""
    sudoku = Sudoku()

    print("\n" + "="*50)
    print("3. PERFORMANCE BENCHMARK:")

    times = []
    for i in range(5):
        puzzle, _ = sudoku.generate_puzzle("medium")

        start_time = time.time()
        solved = sudoku.solve(copy.deepcopy(puzzle))
        end_time = time.time()

        times.append(end_time - start_time)
        print(f"Puzzle {i+1}: {end_time - start_time:.4f} seconds")

    print(f"Average solving time: {sum(times)/len(times):.4f} seconds")

if __name__ == "__main__":
    main()
    benchmark()

=== SUDOKU SOLVER AND GENERATOR ===

1. SOLVING A PUZZLE:
Original Puzzle:
5 3 . | . 7 . | . . . 
6 . . | 1 9 5 | . . . 
. 9 8 | . . . | . 6 . 
---------------------
8 . . | . 6 . | . . 3 
4 . . | 8 . 3 | . . 1 
7 . . | . 2 . | . . 6 
---------------------
. 6 . | . . . | 2 8 . 
. . . | 4 1 9 | . . 5 
. . . | . 8 . | . 7 9 

Solved Puzzle:
5 3 4 | 6 7 8 | 9 1 2 
6 7 2 | 1 9 5 | 3 4 8 
1 9 8 | 3 4 2 | 5 6 7 
---------------------
8 5 9 | 7 6 1 | 4 2 3 
4 2 6 | 8 5 3 | 7 9 1 
7 1 3 | 9 2 4 | 8 5 6 
---------------------
9 6 1 | 5 3 7 | 2 8 4 
2 8 7 | 4 1 9 | 6 3 5 
3 4 5 | 2 8 6 | 1 7 9 

2. GENERATING PUZZLES:

EASY puzzle:
. . . | 3 1 . | 7 9 8 
5 . 1 | 8 . . | 3 2 6 
9 . . | 7 6 . | . 1 . 
---------------------
1 . 7 | . 9 . | 8 . 2 
. . 8 | 1 2 . | . . 7 
4 . 6 | 5 . . | 9 3 . 
---------------------
8 . . | . . . | 6 5 . 
7 . 5 | 9 3 6 | . 8 4 
6 3 2 | 4 . 8 | 1 . 9 
Valid puzzle: True

MEDIUM puzzle:
8 . . | . . 3 | 4 . . 
. . . | 7 8 5 | . 2 . 
3 . 2 | 4 9 6 | . 8 . 
--------------